NA CAMADA SILVER SÃO FEITOS BOA PARTE DOS TRATAMENTOS, ALTERAÇÕES DE NOMES DE COLUNAS E FILTROS


PRIMEIRAMENTE VAMOS IMPORTAR OS NOTEBOOKS QUE ESTÃO PRESENTES NA CAMADA BRONZE


In [0]:
import pyspark.sql.functions as F

In [0]:
df_bronze_dm_cotacao_dolar = spark.table("ecommerce.bronze.dm_cotacao_dolar")
df_bronze_ft_avaliacoes_pedidos = spark.table("ecommerce.bronze.ft_avaliacoes_pedidos")
df_bronze_ft_consumidores = spark.table("ecommerce.bronze.ft_consumidores")
df_bronze_ft_geolocalizacao = spark.table("ecommerce.bronze.ft_geolocalizacao")
df_bronze_ft_itens_pedidos = spark.table("ecommerce.bronze.ft_itens_pedidos")
df_bronze_ft_pagamentos_pedidos = spark.table("ecommerce.bronze.ft_pagamentos_pedidos")
df_bronze_ft_pedidos = spark.table("ecommerce.bronze.ft_pedidos")
df_bronze_ft_produtos = spark.table("ecommerce.bronze.ft_produtos")
df_bronze_ft_vendedoras = spark.table("ecommerce.bronze.ft_vendedores")
df_bronze_dm_categoria_produtos_traducao = spark.table("ecommerce.bronze.dm_categoria_produtos_traducao")


In [0]:
display(df_bronze_ft_consumidores.limit(5))

AGORA IREI ALTERAR OS NOMES DAS MINHAS COLUNAS CONFORME O SOLICITADO

In [0]:
df_silver_ft_consumidores = df_bronze_ft_consumidores.select(
    F.col("customer_id").alias("id_consumidor"),
    F.col("customer_zip_code_prefix").alias("prefixo_cep"),
    F.col('customer_city').alias("cidade"),
    F.col('customer_state').alias("estado")
)

In [0]:
display(
    df_silver_ft_consumidores.select(
        F.count(
            F.when(F.col("id_consumidor").isNull(), 1)
        ).alias("qtd_nulos_id_consumidor")
    )
)

VAMOS CONTAR A QUANTIDADE DE NULOS EXISTENTES E DUPLICADOS NA COLUNA DE ID_CONSUMIDOR

In [0]:


qtd_nulos = df_silver_ft_consumidores.filter(F.col("id_consumidor").isNull()).count()

qtd_duplicados = (
    df_silver_ft_consumidores
    .groupBy("id_consumidor")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Nulos: {qtd_nulos}")
print(f"Duplicados: {qtd_duplicados}")


In [0]:
df_silver_ft_consumidores = (
    df_silver_ft_consumidores
    .withColumn("cidade", F.upper(F.col("cidade")))
    .withColumn("estado", F.upper(F.col("estado")))
)

In [0]:
display(df_silver_ft_consumidores.limit(3))

AGORA FAREMOS AS VERIFICAÇÕES PARA O ft_pedidos

In [0]:
display(df_bronze_ft_pedidos.limit(3))

In [0]:
df_silver_ft_pedidos = df_bronze_ft_pedidos.select(
    F.col("order_id").alias("id_pedido"),
    F.col("customer_id").alias("id_consumidor"),
    F.col("order_status").alias("status"),
    F.col("order_purchase_timestamp").alias("pedido_compra_timestamp"),
    F.col("order_approved_at").alias("pedido_aprovado_timestamp"),
    F.col("order_delivered_carrier_date").alias("pedido_carregado_timestamp"),
    F.col("order_delivered_customer_date").alias("pedido_entregue_timestamp"),
    F.col("order_estimated_delivery_date").alias("pedido_estimativa_entrega_timestamp")
)

In [0]:
qtd_nulos = df_silver_ft_pedidos.filter(F.col("id_consumidor").isNull()).count()

qtd_duplicados = (
    df_silver_ft_pedidos
    .groupBy("id_consumidor")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Nulos: {qtd_nulos}")
print(f"Duplicados: {qtd_duplicados}")



In [0]:
df_silver_ft_pedidos = df_silver_ft_pedidos.withColumn(
    "status",
    F.when(F.col("status") == "canceled", "cancelado")
    .when(F.col("status") == "shipped", "enviado")
    .when(F.col("status") == "processing", "em processamento")
    .when(F.col("status") == "unavailable", "indisponivel")
    .when(F.col("status") == "invoiced", "faturado")
    .when(F.col("status") == "approved", "aprovado")
    .when(F.col("status") == "delivered", "entregue")
    .when(F.col("status") == "created", "criado")
    .otherwise("nao_classificado")
)

In [0]:
display(df_silver_ft_pedidos.limit(5))


VERIFICANDO SE EXISATEM CASOS EM QUE NÃO TENHO CLASSIFICAÇÃO

In [0]:
df_silver_ft_pedidos.filter(F.col("status") == "nao_classificado").display()

In [0]:
df_silver_ft_pedidos.display()

AGORA IREI CRIAR AS COLUNAS NECESSÁRIAS

In [0]:
df_silver_ft_pedidos = (
    df_silver_ft_pedidos
    .withColumn(
        "tempo_entrega_dias",
        F.datediff(F.col("pedido_entregue_timestamp"), F.col("pedido_compra_timestamp"))
    )
    .withColumn(
        "tempo_entrega_estimado_dias",
        F.datediff(F.col("pedido_estimativa_entrega_timestamp"), F.col("pedido_compra_timestamp"))
    )
    .withColumn(
        "diferenca_entrega_dias",
        F.col("tempo_entrega_estimado_dias") - F.col("tempo_entrega_dias")
    )
    .withColumn(
        "entregue_no_prazo",
        F.when(F.col("diferenca_entrega_dias") <= 0, "Sim")
         .when(F.col("diferenca_entrega_dias") > 0, "Não")
         .otherwise("Não Entregue")
    )
)

display(df_silver_ft_pedidos)


AGORA FAREI OS TRATAMENTOS PARA FT_ITENS_PEDIDOS

In [0]:
df_silver_ft_itens_pedidos = df_bronze_ft_itens_pedidos.select(
    F.col("order_id").alias("id_pedido"),
    F.col("order_item_id").alias("id_item"),
    F.col("product_id").alias("id_produto"),
    F.col("seller_id").alias("id_vendedor"),
    F.col("price").alias("preco_BRL"),
    F.col("freight_value").alias("preco_frete")
)


In [0]:
df_silver_ft_itens_pedidos.display()

Analisando se existe alguma coluna vazia ou repetida

AGORA IREMOS TRABALHAR A TABELA DE ft_itens_pedidos

In [0]:
df_bronze_ft_pagamentos_pedidos.printSchema()

In [0]:
display(df_bronze_ft_pagamentos_pedidos.limit(3))

Na parte da tabela está pedidno na atividade para alterar a coluna order_status na tabela de pagamentos, mas está coluna, analisando os valores percebi que estava sendo falado da coluna payment_type e não de order_status

In [0]:
df_silver_ft_pagamentos = (
    df_bronze_ft_pagamentos_pedidos
    .select(
        F.col("order_id").alias("id_pedido"),
        F.col("payment_sequential").alias("codigo_pagamento"),
        F.col("payment_type").alias("forma_pagamento"),
        F.col("payment_installments").alias("parcelas"),
        F.col("payment_value").alias("valor_pagamento")
    )
    .withColumn(
        "forma_pagamento",
        F.when(F.col("forma_pagamento") == "credit_card", "Cartão de Crédito")
        .when(F.col("forma_pagamento") == "boleto", "Boleto")
        .when(F.col("forma_pagamento") == "voucher", "Voucher")
        .when(F.col("forma_pagamento") == "debit_card", "Cartão de Débito")
        .otherwise("Outro")
    )
)

display(df_silver_ft_pagamentos.limit(3))


AGORA FAREMOS O TRATAMENTO PARA A TABELA ft_avaliacoes_pedidos

In [0]:
display(df_bronze_ft_avaliacoes_pedidos)

irei utilizar regex para fazer filtros e vou fazer uma verificação de tipagem de algumas colunas

In [0]:
tamanho_antes = df_bronze_ft_avaliacoes_pedidos.count()

In [0]:
df_silver_ft_avaliacoes_pedidos = df_bronze_ft_avaliacoes_pedidos.filter(F.col("review_id").rlike("^[a-f0-9]{32}$"))
#USEI ESSE REGEX, POIS O ID ESTÁ NO PADRÃO DE TER 9 CARACTERES DE A-F E 0-9 E TAMANHO DE 32 
tamanho_depois = df_silver_ft_avaliacoes_pedidos.count()

In [0]:
qnt_removida = tamanho_antes - tamanho_depois
print(qnt_removida)

In [0]:
df_silver_ft_avaliacoes_pedidos = df_silver_ft_avaliacoes_pedidos.filter(
    F.col("review_creation_date").rlike(r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$") &
    F.col("review_answer_timestamp").rlike(r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$")
                                                                   )

### AQUI EU DEFINI O REGEX PARA O FORMATO DE 4 DÍGITOS DE ANO, 2 DE MÊS, 2 DE DIA, 2 DE HORA, 2 DE MINUTO E 2 DE SEGUNDO 


In [0]:
print(qnt_removida)

# PERCEBI QUE NÃO MUDOU NADA, OU SEJA TODOS QUE ESTAVAM COM O CAMPO ID PREENCHIDO INCORRTAMENTE ESTAVAM COM OUTROS CAMPOS, COMO OS DE HORA PREENCHIDOS INCORRETAMENTE TAMBÉM

In [0]:
df_silver_ft_avaliacoes_pedidos = df_silver_ft_avaliacoes_pedidos.select(
    F.col("review_id").alias("id_avaliacao"),
    F.col("order_id").alias("id_pedido"),
    F.col("review_score").alias("avaliacao"),
    F.col("review_creation_date").alias("data_avaliacao"),
    F.col("review_answer_timestamp").alias("data_resposta"),
    F.col("review_comment_title").alias("titulo_avaliacao"),
    F.col("review_comment_message").alias("comentario_avaliacao")
)



In [0]:
df_silver_ft_avaliacoes_pedidos.printSchema()

AGORA FAREMOS PARA PRODUTO

In [0]:
df_silver_ft_produtos = df_bronze_ft_produtos.select(
    F.col("product_id").alias("id_produto"),
    F.col("product_category_name").alias("categoria_produto"),
    F.col("product_weight_g").alias("peso_produto_gramas"),
    F.col("product_length_cm").alias("comprimento_centimetros"),
    F.col("product_height_cm").alias("altura_centimetros"),
    F.col("product_width_cm").alias("largura_centimetros") 
)

AGORA FAREMOS PARA ft_vendedores

In [0]:
df_silver_ft_vendedores = (
    df_bronze_ft_vendedoras
    .withColumn("cidade", F.upper(F.col("seller_city")))
    .withColumn("estado", F.upper(F.col("seller_state")))
    .select(
        F.col("seller_id").alias("id_vendedor"),
        F.col("seller_zip_code_prefix").alias("prefixo_cep"),
        F.col("cidade"),
        F.col("estado")
    )
)

AGORA FAREMOS PARA dm_categoria_produtos_traducao

In [0]:
df_silver_dm_categoria_produtos_traducao = df_bronze_dm_categoria_produtos_traducao.select(
    F.col("product_category_name").alias("nome_produto_pt"),
    F.col("product_category_name_english").alias("nome_produto_en")
)

AGORA FAREMOS PARA COTACAO_DOLAR

In [0]:
df_bronze_dm_cotacao_dolar.display()

In [0]:
from pyspark.sql import Window

#passei a o timestamp para formato de aaaa-mm-dd
df_base = df_bronze_dm_cotacao_dolar.withColumn(
    "data", F.to_date("dataHoraCotacao")
)

datas = (
    df_base
    .select(F.sequence(F.min("data"), F.max("data")).alias("datas"))
    .select(F.explode("datas").alias("data"))
)

#
df_com_datas = (
    datas
    .join(df_base.select("data", "cotacaoCompra"), on="data", how="left")
    .orderBy("data")
)

window_spec = Window.orderBy("data").rowsBetween(Window.unboundedPreceding, 0)
df_silver_dm_cotacao_dolar = (
    df_com_datas
    .withColumn("cotacao_dolar", F.last("cotacaoCompra", ignorenulls=True).over(window_spec))
    .select("data", "cotacao_dolar")
)

display(df_silver_dm_cotacao_dolar)


irei dropar as tables da silver



In [0]:
%sql
DROP TABLE IF EXISTS ecommerce.silver.ft_pedidos;
DROP TABLE IF EXISTS ecommerce.silver.ft_consumidores;
DROP TABLE IF EXISTS ecommerce.silver.ft_itens_pedidos;
DROP TABLE IF EXISTS ecommerce.silver.ft_pagamentos;
DROP TABLE IF EXISTS ecommerce.silver.ft_avaliacoes_pedidos;
DROP TABLE IF EXISTS ecommerce.silver.ft_produtos;
DROP TABLE IF EXISTS ecommerce.silver.ft_vendedores;
DROP TABLE IF EXISTS ecommerce.silver.dm_categoria_produtos_traducao;
DROP TABLE IF EXISTS ecommerce.silver.dm_cotacao_dolar;

AGORA IREI FAZER O CAST DOS MEUS DADOS

In [0]:
# CONSUMIDORES
df_silver_ft_consumidores = (df_silver_ft_consumidores
    .withColumn("id_consumidor", F.col("id_consumidor").cast("string"))
    .withColumn("prefixo_cep", F.col("prefixo_cep").cast("int"))
    .withColumn("cidade", F.col("cidade").cast("string"))
    .withColumn("estado", F.col("estado").cast("string"))
)

#  PEDIDOS 
df_silver_ft_pedidos = (df_silver_ft_pedidos
    .withColumn("id_pedido", F.col("id_pedido").cast("string"))
    .withColumn("id_consumidor", F.col("id_consumidor").cast("string"))
    .withColumn("status", F.col("status").cast("string"))
    .withColumn("pedido_compra_timestamp", F.col("pedido_compra_timestamp").cast("timestamp"))
    .withColumn("pedido_aprovado_timestamp", F.col("pedido_aprovado_timestamp").cast("timestamp"))
    .withColumn("pedido_carregado_timestamp", F.col("pedido_carregado_timestamp").cast("timestamp"))
    .withColumn("pedido_entregue_timestamp", F.col("pedido_entregue_timestamp").cast("timestamp"))
    .withColumn("pedido_estimativa_entrega_timestamp", F.col("pedido_estimativa_entrega_timestamp").cast("timestamp"))
    .withColumn("tempo_entrega_dias", F.col("tempo_entrega_dias").cast("int"))
    .withColumn("tempo_entrega_estimado_dias", F.col("tempo_entrega_estimado_dias").cast("int"))
    .withColumn("diferenca_entrega_dias", F.col("diferenca_entrega_dias").cast("int"))
    .withColumn("entregue_no_prazo", F.col("entregue_no_prazo").cast("string"))
)

# ITENS DE PEDIDOS 
df_silver_ft_itens_pedidos = (df_silver_ft_itens_pedidos
    .withColumn("id_pedido", F.col("id_pedido").cast("string"))
    .withColumn("id_item", F.col("id_item").cast("int"))
    .withColumn("id_produto", F.col("id_produto").cast("string"))
    .withColumn("id_vendedor", F.col("id_vendedor").cast("string"))
    .withColumn("preco_BRL", F.col("preco_BRL").cast("decimal(12,2)"))
    .withColumn("preco_frete", F.col("preco_frete").cast("decimal(12,2)"))
)

#PAGAMENTOS
df_silver_ft_pagamentos = (df_silver_ft_pagamentos
    .withColumn("id_pedido", F.col("id_pedido").cast("string"))
    .withColumn("codigo_pagamento", F.col("codigo_pagamento").cast("int"))
    .withColumn("forma_pagamento", F.col("forma_pagamento").cast("string"))
    .withColumn("parcelas", F.col("parcelas").cast("int"))
    .withColumn("valor_pagamento", F.col("valor_pagamento").cast("decimal(12,2)"))
)

#  AVALIAÇÕES
df_silver_ft_avaliacoes_pedidos = (df_silver_ft_avaliacoes_pedidos
    .withColumn("id_avaliacao", F.col("id_avaliacao").cast("string"))
    .withColumn("id_pedido", F.col("id_pedido").cast("string"))
    .withColumn("avaliacao", F.col("avaliacao").cast("int"))
    .withColumn("data_avaliacao", F.to_timestamp("data_avaliacao", "yyyy-MM-dd HH:mm:ss"))
    .withColumn("data_resposta", F.to_timestamp("data_resposta", "yyyy-MM-dd HH:mm:ss"))
    .withColumn("titulo_avaliacao", F.col("titulo_avaliacao").cast("string"))
    .withColumn("comentario_avaliacao", F.col("comentario_avaliacao").cast("string"))
)

# PRODUTOS
df_silver_ft_produtos = (df_silver_ft_produtos
    .withColumn("id_produto", F.col("id_produto").cast("string"))
    .withColumn("categoria_produto", F.col("categoria_produto").cast("string"))
    .withColumn("peso_produto_gramas", F.col("peso_produto_gramas").cast("int"))
    .withColumn("comprimento_centimetros", F.col("comprimento_centimetros").cast("int"))
    .withColumn("altura_centimetros", F.col("altura_centimetros").cast("int"))
    .withColumn("largura_centimetros", F.col("largura_centimetros").cast("int"))
)

# VENDEDORES 
df_silver_ft_vendedores = (df_silver_ft_vendedores
    .withColumn("id_vendedor", F.col("id_vendedor").cast("string"))
    .withColumn("prefixo_cep", F.col("prefixo_cep").cast("int"))
    .withColumn("cidade", F.col("cidade").cast("string"))
    .withColumn("estado", F.col("estado").cast("string"))
)

# TRADUÇÃO CATEGORIAS 
df_silver_dm_categoria_produtos_traducao = (df_silver_dm_categoria_produtos_traducao
    .withColumn("nome_produto_pt", F.col("nome_produto_pt").cast("string"))
    .withColumn("nome_produto_en", F.col("nome_produto_en").cast("string"))
)

# COTAÇÃO DÓLAR
df_silver_dm_cotacao_dolar = (df_silver_dm_cotacao_dolar
    .withColumn("data", F.col("data").cast("date"))
    .withColumn("cotacao_dolar", F.col("cotacao_dolar").cast("decimal(12,2)"))
)


CARREGAMENTO NA CAMADA SILVER

In [0]:
from pyspark.sql.functions import current_timestamp

def salvar_silver(df, nome):
    df_final = df.withColumn("data_ingestao", F.current_timestamp())
    (
        df_final
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true") 
        .saveAsTable(f"ecommerce.silver.{nome}")
    )
    print(f"Tabela ecommerce.silver.{nome} salva com sucesso com coluna 'data_ingestao'.")



In [0]:
salvar_silver(df_silver_ft_consumidores, "ft_consumidores")
salvar_silver(df_silver_ft_pedidos, "ft_pedidos")
salvar_silver(df_silver_ft_itens_pedidos, "ft_itens_pedidos")
salvar_silver(df_silver_ft_pagamentos, "ft_pagamentos")
salvar_silver(df_silver_ft_avaliacoes_pedidos, "ft_avaliacoes_pedidos")
salvar_silver(df_silver_ft_produtos, "ft_produtos")
salvar_silver(df_silver_ft_vendedores, "ft_vendedores")
salvar_silver(df_silver_dm_categoria_produtos_traducao, "dm_categoria_produtos_traducao")
salvar_silver(df_silver_dm_cotacao_dolar, "dm_cotacao_dolar")


VALIDAÇÕES


In [0]:
pedidos_orfaos = df_silver_ft_pedidos.join(
    df_silver_ft_consumidores,
    on="id_consumidor",
    how="left_anti"
)

qtd_pedidos_orfaos = pedidos_orfaos.count()

print(f"Quantidade de pedidos órfãos (sem consumidor): {qtd_pedidos_orfaos}")

if qtd_pedidos_orfaos > 0:
    df_silver_ft_pedidos = df_silver_ft_pedidos.join(
        df_silver_ft_consumidores,
        on="id_consumidor",
        how="inner"
    )


In [0]:
itens_orfaos = df_silver_ft_itens_pedidos.join(
    df_silver_ft_pedidos,
    on="id_pedido",
    how="left_anti"
)

qtd_itens_orfaos = itens_orfaos.count()

print(f"Quantidade de itens de pedidos órfãos (sem pedido): {qtd_itens_orfaos}")

if qtd_itens_orfaos > 0:
    df_silver_ft_itens_pedidos = df_silver_ft_itens_pedidos.join(
        df_silver_ft_pedidos,
        on="id_pedido",
        how="inner"
    )


In [0]:
qtd_pedidos_orfaos = pedidos_orfaos.count()
qtd_itens_orfaos  = itens_orfaos.count()


In [0]:
if qtd_pedidos_orfaos > 0:
    df_silver_ft_pedidos = df_silver_ft_pedidos.join(
        df_silver_ft_consumidores,
        on="id_consumidor",
        how="inner"
    )
    df_silver_ft_pedidos.write.format("delta").mode("overwrite").saveAsTable("ecommerce.silver.ft_pedidos")

In [0]:
if qtd_itens_orfaos > 0:
    df_silver_ft_itens_pedidos = df_silver_ft_itens_pedidos.join(
        df_silver_ft_pedidos,
        on="id_pedido",
        how="inner"
    )
    df_silver_ft_itens_pedidos.write.format("delta").mode("overwrite").saveAsTable("ecommerce.silver.ft_itens_pedidos")

In [0]:
df_pedidos = spark.table("ecommerce.bronze.ft_pedidos")
df_pagamentos = spark.table("ecommerce.bronze.ft_pagamentos_pedidos")
df_consumidores = spark.table("ecommerce.bronze.ft_consumidores")
df_cotacao = spark.table("ecommerce.bronze.dm_cotacao_dolar")

df_pedidos = df_pedidos.withColumn(
    "order_purchase_timestamp",
    F.to_timestamp("order_purchase_timestamp")
).withColumn(
    "data_pedido", F.to_date("order_purchase_timestamp")
)

df_cotacao = (
    df_cotacao
    .withColumn("dataHoraCotacao", F.to_timestamp("dataHoraCotacao"))
    .withColumn("dataCotacao", F.to_date("dataHoraCotacao"))
    .withColumnRenamed("cotacaoCompra", "cotacao_dolar")
)

df_pagamentos_agg = (
    df_pagamentos.groupBy("order_id")
    .agg(F.sum("payment_value").alias("valor_total_pago_brl"))
)

df_pedido_total = (
    df_pedidos
    .join(df_pagamentos_agg, on="order_id", how="left")
    .join(df_consumidores, "customer_id", "left")
)

df_pedido_total = (
    df_pedido_total.join(
        df_cotacao.select("dataCotacao", "cotacao_dolar"),
        df_pedido_total.data_pedido == df_cotacao.dataCotacao,
        "left"
    )
)

df_pedido_total = df_pedido_total.withColumn(
    "valor_total_pago_usd",
    F.when(
        F.col("cotacao_dolar").isNotNull(),
        F.col("valor_total_pago_brl") / F.col("cotacao_dolar")
    )
)

df_silver_ft_pedido_total = df_pedido_total.select(
    F.col("data_pedido").alias("data"),
    F.col("order_id").alias("id_pedido"),
    F.col("customer_id").alias("id_consumidor"),
    F.col("order_status").alias("status"),
    F.round("valor_total_pago_brl", 2).alias("valor_total_pago_brl"),
    F.round("valor_total_pago_usd", 2).alias("valor_total_pago_usd")
)

df_silver_ft_pedido_total.display()

#não irei salvar essa tabela na silver pq n foi solicitado